In [3]:
'''Try a support vector machine regressor (sklearn.svm.SVR) with various hyperparameters, such as kernel="linear" (with various values for 
 the C hyperparameter) or kernel="rbf" (with various values for the C and gamma hyperparameters). 
 Note that support vector machines don’t scale well to large datasets, so you should probably train your model on just the first 5,000 instances 
 of the training set and use only 3-fold cross-validation, or else it will take hours. Don’t worry about what the hyperparameters mean for now. '''

'Try a support vector machine regressor (sklearn.svm.SVR) with various hyperparameters, such as kernel="linear" (with various values for \n the C hyperparameter) or kernel="rbf" (with various values for the C and gamma hyperparameters). \n Note that support vector machines don’t scale well to large datasets, so you should probably train your model on just the first 5,000 instances \n of the training set and use only 3-fold cross-validation, or else it will take hours. Don’t worry about what the hyperparameters mean for now. '

In [ ]:
# importing the basic libaries we need 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import fetch_openml

ames = fetch_openml(name='house_prices', as_frame=True)

print(ames.keys())
print()
print(ames.DESCR)                   

In [ ]:
df = ames.frame

print(df.info())
print()

df.head(5)

In [ ]:
num_cols=df.select_dtypes(include=['int64', 'float64']).columns
cat_cols=df.select_dtypes(include=['str', 'object']).columns

print('Numeric:', len(num_cols),'\n', num_cols)
print()
print('Categorical:', len(cat_cols), '\n', cat_cols)
print()
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
n = len(num_cols)          # number of numeric columns
cols = 3                   # number of plots per row
rows = math.ceil(n / cols) # number of rows needed

# Set figure size proportional to rows and columns
fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*4))
axes = axes.flatten()

for col, ax in zip(num_cols, axes):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} (skew={df[col].skew():.2f})')

# Hide unused axes
for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
skewness = df[num_cols].skew()
skewed_cols = skewness[skewness > 1].index.to_list()

print(skewed_cols)
print()
print(skewness)

In [ ]:
# using Boxplot to detect outliers

n= len(num_cols)
cols = 3
rows = math.ceil(n / cols)


fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*4))
axes = axes.flatten()

for col, ax in zip(num_cols, axes):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f'Distribution {col}: (skew={df[col].skew():.2f})')

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
outlier_cols = []
no_outlier_cols = []
outlier_summary = []

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    if len(outliers) > 0:
        outlier_cols.append(col)   
    else:
        no_outlier_cols.append(col)
    
    outlier_summary.append({
        'Column': col,
        'Outlier Count': len(outliers),
        'Total Rows': len(df),
        'Outlier %': round(len(outliers)/len(df) * 100, 2)
    })

print("Columns with outliers:", outlier_cols)
print()
print("Columns without outliers:", no_outlier_cols)
print()

outlier_table = pd.DataFrame(outlier_summary)
display(outlier_table.sort_values(by='Outlier Count', ascending=False))

In [ ]:
df_corr = df.corr(numeric_only=True)
print(df_corr['SalePrice'].sort_values(ascending=False))

# Strong predictors: Typically ≥ 0.5 with the target
# Moderate predictors: Roughly 0.1 – 0.4.
# Weak predictors: Less than 0.1 (or negative correlations).

# Strong predictors → keep and possibly transform for skewness.
# Moderate predictors → keep if they add unique signal, but consider dropping if they overlap too much with stronger features.
# Weak predictors → usually drop, unless domain knowledge(real-world context and expertise about the dataset or the problem you’re solving) 
# says they matter e.g., PoolArea could matter in luxury housing markets even if correlation is low overall

In [ ]:
# Keep and transform strong predictors mGrLivArea, TotalBsmtSF, 1stFlrSF, GarageArea, LotFrontage, MasVnrArea.
# Apply log transformation or winsorization to reduce skewness while preserving predictive signal.
# Drop or simplify weak predictors MiscVal, 3SsnPorch, PoolArea, KitchenAbvGr, EnclosedPorch.
# They have high outlier % and low correlation → dropping them will simplify the model without losing much accuracy.
# Alternatively, convert to binary flags (e.g., “Has Pool” yes/no).
# Categorical-like numeric features MSSubClass, KitchenAbvGr, BsmtHalfBath.
# Treat as categorical variables instead of continuous. Outliers here are just rare categories.
# SalePrice (target) Skew = 1.88 → apply log transformation before modeling. This stabilizes variance and improves regression performance.